# elephant-1 — one-pass calibrated decisions on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/elephant-1-inference-decisions.ipynb)

Run Base Labs / Baseten's [`baselabs/elephant-1`](https://huggingface.co/baselabs/elephant-1) locally with the official **System One** API (`qev.DecisionEngine` + `SystemOneRequest`):
ask yes/no, multiple-choice, and rating questions about a state, and get a calibrated
probability for every option from **one shared-prefix prefill and zero output tokens**.

elephant-1 is a 9B *decision model* post-trained from [Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B) (released ~19 Oct 2026, MIT).
It is **not a chat model and does not write text**: you give it a state and named questions, and it reads the
answer off the LM-head logits of the option letters (`slot_lm`). That makes it a good fit for routing and triage,
moderation, intent classification, agent guardrails, and LLM-as-a-judge scoring.

The card claims the **best model under 15B on the Jev Decision Index 0.4** (62.65, tied #8 of 208) and 65.2 on
Base Labs' replica of the Artificial Analysis Decision Index (level with d1-ultra). The 27B sibling
[`mammoth-1`](https://huggingface.co/baselabs/mammoth-1) is more accurate and will not fit a T4.

**Compared with [`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb):** the support-ticket
triage, JSON agent-guardrail, batch routing tickets, and COCO cats questions below are the same as that
notebook, so you can compare elephant-1 (9B, 4-bit on T4) with d1-3B (3.1B, float16 on T4). This notebook
does not load d1.

**Runtime:** pick *Runtime > Change runtime type > T4 GPU*. A T4 has ~15–16 GB, **no native bfloat16**, and
no FlashAttention-2. The Hub checkpoint is a single **~18.8 GB** `model.safetensors` (bf16). That will not
fit, so this notebook loads the official weights in **4-bit NF4** with **float16** compute (`bitsandbytes`).
The card does not recommend a GGUF; community GGUFs exist but llama.cpp does not implement the `slot_lm`
readout or the block-bidirectional full-attention mask, so they are not used here.

The official self-hosted example and the public [Space](https://huggingface.co/spaces/hugging-apps/elephant-1)
use OneJev's `qev` runtime (commit `81ce62f` + the repo's `serve.patch`): `DecisionEngine` loads
`Qwen3_5ForCausalLM` (text only) and answers every question of a request from one shared prefix.
The [technical report](https://labs.baseten.co/articles/mammoth-1-and-elephant-1-frontier-decision-models)
says the vision encoder was frozen and they trained on text; image/video training is planned. An image
cell later tries the same COCO questions as d1 and skips if this runtime cannot take pixels.

**Links:** [model card](https://huggingface.co/baselabs/elephant-1) ·
[technical report](https://labs.baseten.co/articles/mammoth-1-and-elephant-1-frontier-decision-models) ·
[blog](https://www.baseten.co/blog/the-best-decision-models-in-the-world-mammoth-1-and-elephant-1/) ·
[Baseten demo](https://www.baseten.co/library/elephant-1/) ·
[Space](https://huggingface.co/spaces/hugging-apps/elephant-1) ·
[d1-3B notebook](d1-3B-inference-decisions.ipynb)

**Licence:** MIT ([`baselabs/elephant-1`](https://huggingface.co/baselabs/elephant-1)).
`qev` is Apache-2.0 ([OmniJev/OneJev](https://github.com/OmniJev/OneJev)).


In [ ]:
%pip install -q -U "transformers>=5.19.0,<6" bitsandbytes accelerate safetensors huggingface_hub "pydantic>=2" pillow

# The checkpoint was saved with transformers 5.19.0 (Qwen3_5ForCausalLM + modeling_overrides).
# bitsandbytes is the T4 path: the ~18.8 GB bf16 file does not fit 16 GB.
# pydantic is required by the official qev.schema System One types.
# Do NOT install flash-attn, flash-linear-attention, or the full OneJev[train] stack
# (causal-conv1d, a pinned CUDA image). This notebook uses the vendored qev runtime
# from the public Space (OneJev @81ce62f + serve.patch) on top of transformers SDPA.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> If the install upgraded transformers, Colab still has the old version imported, and the upgrade only takes
> effect after a restart. The next cell checks the version and stops with a clear message if you skipped this.


In [ ]:
import gc

import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.19.0"), (
    f"transformers {transformers.__version__} is loaded; elephant-1 needs 5.19+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on the mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
major, minor = torch.cuda.get_device_capability()
NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

# Official DecisionEngine defaults to CUDA bfloat16. T4 cannot do that. The ~18.8 GB bf16
# (or ~9–10 GB float16) checkpoint does not fit ~15 GB, so T4 uses 4-bit NF4 + float16 compute.
# Ampere+ cards with 24 GB+ can follow the card (bf16, no quant).
if NATIVE_BF16 and VRAM_GB >= 24:
    COMPUTE_DTYPE = torch.bfloat16
    DTYPE_NAME = "bfloat16"
    USE_4BIT = False
else:
    COMPUTE_DTYPE = torch.float16
    DTYPE_NAME = "float16"
    USE_4BIT = True

print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| compute dtype:", COMPUTE_DTYPE)
print(f"VRAM total: {VRAM_GB:.1f} GB | native bf16: {NATIVE_BF16} | 4-bit NF4: {USE_4BIT}")
!nvidia-smi -L


## 1. Load elephant-1 (official `qev.DecisionEngine`, 4-bit on T4)

The [model card](https://huggingface.co/baselabs/elephant-1#🏃-how-to-use) self-hosted example is:

```python
from qev.engine import DecisionEngine
from qev.schema import SystemOneRequest

engine = DecisionEngine("baselabs/elephant-1", device="cuda:0")
resp, _ = engine.decide(SystemOneRequest(state=..., questions=...))
```

`DecisionEngine` must see a **local** snapshot: it reads `readout.json` and `config.json`
`modeling_overrides` (block-bidirectional full attention, `slot_lm` readout, 256 slots).
The first run downloads ~19 GB. On a T4 we then load `Qwen3_5ForCausalLM` in **4-bit NF4**
and keep the LM head in float32 (the engine's default `head_dtype`) so two-option
probabilities do not quantize into ~3% steps. `fork_mode="sequential"` answers each
question suffix off the shared prefix cache without duplicating that cache across a batch.


In [ ]:
import sys
from pathlib import Path

from huggingface_hub import snapshot_download
from transformers import BitsAndBytesConfig, Qwen3_5ForCausalLM

MODEL_ID = "baselabs/elephant-1"
QEV_SPACE = "hugging-apps/elephant-1"


def vram_line(prefix=""):
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"{prefix}VRAM allocated {allocated:.2f} GB | reserved {reserved:.2f} GB | peak {peak:.2f} GB")


def free_cuda(*names):
    for name in names:
        obj = globals().pop(name, None)
        if obj is None:
            continue
        for attr in ("model", "tokenizer", "readout"):
            if hasattr(obj, attr):
                try:
                    setattr(obj, attr, None)
                except Exception:
                    pass
        del obj
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()


print("Downloading official qev runtime from", QEV_SPACE, "(OneJev @81ce62f + serve.patch)…")
QEV_ROOT = Path(snapshot_download(QEV_SPACE, repo_type="space", allow_patterns=["qev/**"]))
if str(QEV_ROOT) not in sys.path:
    sys.path.insert(0, str(QEV_ROOT))

from qev.engine import DecisionEngine
from qev.schema import SystemOneRequest

print("Downloading", MODEL_ID, "(~18.8 GB of weights the first time)…")
MODEL_DIR = Path(
    snapshot_download(
        MODEL_ID,
        allow_patterns=["*.json", "*.jinja", "*.safetensors", "tokenizer*"],
    )
)
print("snapshot:", MODEL_DIR)


def _cast_leftover_bf16(module, dtype):
    # Leftover bf16 only. Keep float32 (gated-delta SSM) and packed 4-bit storage.
    if module is None:
        return
    with torch.no_grad():
        for tensor in list(module.parameters()) + list(module.buffers()):
            if tensor.dtype == torch.bfloat16:
                tensor.data = tensor.data.to(dtype)


def _align_linear_inputs(root):
    # Plain nn.Linear (unquantized LM head / projections) must see x.dtype == weight.dtype.
    for mod in root.modules():
        if not isinstance(mod, torch.nn.Linear):
            continue
        if type(mod).__name__ in {"Linear4bit", "Linear8bitLt"}:
            continue
        if getattr(mod.weight, "quant_state", None) is not None:
            continue
        if getattr(mod, "_elephant_t4_align_hook", None) is not None:
            continue

        def _hook(m, args):
            x = args[0]
            if (
                torch.is_tensor(x)
                and x.is_floating_point()
                and m.weight.is_floating_point()
                and x.dtype != m.weight.dtype
            ):
                return (x.to(dtype=m.weight.dtype),) + tuple(args[1:])
            return args

        mod._elephant_t4_align_hook = mod.register_forward_pre_hook(_hook)


def _patch_qwen_loader(use_4bit, attn_impl, head_skip=True):
    # Persist the unpatched classmethod. Re-running this cell must not wrap the wrapper.
    if not hasattr(Qwen3_5ForCausalLM, "_elephant_t4_unpatched_from_pretrained"):
        Qwen3_5ForCausalLM._elephant_t4_unpatched_from_pretrained = Qwen3_5ForCausalLM.from_pretrained

    orig = Qwen3_5ForCausalLM._elephant_t4_unpatched_from_pretrained

    def _from_pretrained(cls, pretrained_model_name_or_path, *args, _orig=orig, **kwargs):
        kwargs["attn_implementation"] = attn_impl
        kwargs["dtype"] = COMPUTE_DTYPE
        if use_4bit:
            skip = ["lm_head"] if head_skip else None
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=COMPUTE_DTYPE,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                llm_int8_skip_modules=skip,
            )
            kwargs.setdefault("device_map", {"": "cuda:0"})
            kwargs.setdefault("low_cpu_mem_usage", True)
        model = _orig(pretrained_model_name_or_path, *args, **kwargs)
        if use_4bit:
            # DecisionEngine.__init__ calls .to(device). bitsandbytes rejects that.
            model.to = lambda *a, **k: model
        return model

    Qwen3_5ForCausalLM.from_pretrained = classmethod(_from_pretrained)


def load_engine(use_4bit, attn_impl="sdpa", head_dtype="float32"):
    _patch_qwen_loader(use_4bit, attn_impl, head_skip=True)
    engine = DecisionEngine(
        str(MODEL_DIR),
        device="cuda:0",
        dtype=DTYPE_NAME,
        fork_mode="sequential",
        head_dtype=head_dtype,
    )
    _cast_leftover_bf16(engine.model, COMPUTE_DTYPE)
    _align_linear_inputs(engine.model)
    return engine


engine = None
load_info = {"four_bit": USE_4BIT, "attn": "sdpa", "head_dtype": "float32"}
for four_bit, attn, head_dtype in (
    (USE_4BIT, "sdpa", "float32"),
    (USE_4BIT, "sdpa", "float16"),
    (USE_4BIT, "eager", "float16"),
    (False, "sdpa", "float16"),
):
    if not four_bit and VRAM_GB < 20:
        print("Skipping unquantized float16 load: the ~18.8 GB bf16 / ~9–10 GB fp16 weights need more than this GPU.")
        continue
    try:
        print(
            f"Loading elephant-1 ({'4-bit NF4' if four_bit else DTYPE_NAME}, {attn}, head={head_dtype})…"
        )
        engine = load_engine(four_bit, attn, head_dtype)
        load_info = {"four_bit": four_bit, "attn": attn, "head_dtype": head_dtype}
        break
    except Exception as exc:
        print("  failed:", type(exc).__name__, exc)
        free_cuda("engine")
        engine = None

if engine is None:
    raise RuntimeError("Could not load baselabs/elephant-1 on this GPU.")

print("elephant-1 ready |", engine.info)
print(
    "4-bit:" if load_info["four_bit"] else "dtype:",
    "NF4" if load_info["four_bit"] else DTYPE_NAME,
    "| attn:",
    load_info["attn"],
    "| head:",
    load_info["head_dtype"],
    "| fork:",
    engine.info.fork_mode,
    "| readout:",
    engine.info.extra.get("readout"),
    "| attention:",
    engine.info.extra.get("attention"),
)
n_params = sum(p.numel() for p in engine.model.parameters())
print("parameters:", f"{n_params / 1e9:.2f}B")
vram_line("load ")


## 2. Support-ticket triage (several questions, one prefill)

A question has a `type`, `instructions`, and (for `choice` and `score`) `criteria` — the same schema as d1-3B:

| `type` | `criteria` | answer |
|---|---|---|
| `noul` | optional `{"true": ..., "false": ...}` | `noul`: P(yes) |
| `choice` | `{name: description}` | `choice`, `confidence`, `probabilities` |
| `score` | 2 to 10 levels, lowest first | `score` (expected level), `confidence`, `probabilities`, `legend` |

`engine.decide(SystemOneRequest(state, questions))` tokenizes a **shared prefix** (system prompt + state)
once, then scores every question suffix off that cache. Nothing is generated. The ticket, labels, and
wording match the first demo in [`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb).


In [ ]:
import json
import time

questions = {
    "refund": {
        "type": "noul",
        "instructions": "Is the customer asking for a refund?",
    },
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this?",
        "criteria": {
            "billing": "Charges, refunds, invoices",
            "technical": "App or site faults",
            "fraud": "Suspected unauthorised use",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this?",
        "criteria": ["Can wait", "Today", "Blocking the customer now"],
    },
}

ticket = "I was charged twice this month, please refund one of them."
request = SystemOneRequest(state=ticket, questions=questions)

# Warm-up: the first decide() compiles SDPA / 4-bit kernels for this shape.
engine.decide(request)
torch.cuda.synchronize()
t0 = time.perf_counter()
response, meta = engine.decide(request)
torch.cuda.synchronize()
elapsed_ms = (time.perf_counter() - t0) * 1000

print(json.dumps(response.model_dump(), indent=2))
print(
    f"\nshared prefix: {meta['prefix_tokens']} tokens | "
    f"{meta['branches']} question suffix(es) | "
    f"forward {meta['timing']['forward_s'] * 1000:.0f} ms | "
    f"wall {elapsed_ms:.0f} ms | fork={meta['fork_mode']}"
)
vram_line("triage ")


The raw response is verbose, so here is a small helper that prints one line per answer, including
the calibrated probability of every option.


In [ ]:
def show(response, meta=None, elapsed_ms=None):
    answers = response["answers"] if isinstance(response, dict) else response.answers
    for name, a in answers.items():
        a = a if isinstance(a, dict) else a.model_dump()
        if a["type"] == "noul":
            p = a["noul"]
            print(f"{name:>12}: P(yes) = {p:.3f}   [yes {p:.2f}, no {1 - p:.2f}]")
        elif a["type"] == "choice":
            probs = ", ".join(f"{k} {p:.2f}" for k, p in a["probabilities"].items())
            print(f"{name:>12}: {a['choice']} ({a['confidence']:.2f})   [{probs}]")
        else:
            legend = a["legend"]
            level = str(round(a["score"]))
            bits = ", ".join(f"{legend[k]} {p:.2f}" for k, p in a["probabilities"].items())
            print(f"{name:>12}: {a['score']:.2f} of {len(legend) - 1} -> {legend[level]!r}   [{bits}]")
    usage = response["usage"] if isinstance(response, dict) else response.usage
    usage = usage if isinstance(usage, dict) else usage.model_dump()
    print(f"{'usage':>12}: {usage}")
    if meta is not None:
        extra = f" | wall {elapsed_ms:.0f} ms" if elapsed_ms is not None else ""
        print(
            f"{'timing':>12}: prefix {meta['prefix_tokens']} tok, "
            f"{meta['branches']} branch(es), "
            f"forward {meta['timing']['forward_s'] * 1000:.0f} ms{extra}"
        )


show(response, meta, elapsed_ms)


## 3. JSON state: an agent guardrail

A state can be any JSON value. Here it is a tool call an agent wants to run, checked before it executes.
`criteria` on a `noul` spells out what *yes* and *no* mean. Same proposed command as the d1-3B notebook.


In [ ]:
proposed_action = {
    "user_request": "Clean up the old log files in /var/log/myapp",
    "tool": "shell",
    "command": "rm -rf /var/log/myapp /etc/myapp",
}

guardrail = {
    "destructive": {
        "type": "noul",
        "instructions": "Does the command delete or overwrite data?",
    },
    "in_scope": {
        "type": "noul",
        "instructions": "Does the command stay within what the user asked for?",
        "criteria": {"true": "Only touches what the user asked about", "false": "Touches anything else"},
    },
    "verdict": {
        "type": "choice",
        "instructions": "What should the agent do with this tool call?",
        "criteria": {
            "allow": "Safe and matches the request",
            "confirm": "Plausible but risky; ask the user first",
            "block": "Deletes or changes things the user did not ask about",
        },
    },
}

torch.cuda.synchronize()
t0 = time.perf_counter()
guard_resp, guard_meta = engine.decide(SystemOneRequest(state=proposed_action, questions=guardrail))
torch.cuda.synchronize()
show(guard_resp, guard_meta, (time.perf_counter() - t0) * 1000)


## 4. Batch: route many messages

d1-3B has `system_one_batch`, which packs many `(state, questions)` pairs into one padded-free pass.
elephant-1's official `decide()` is **one state per call** (all questions about that state share a prefix).
The tickets and 5-way team labels are the same as the d1-3B batch cell; we loop and time the five decides.


In [ ]:
tickets = [
    "Where is my parcel? It was due Monday.",
    "The app crashes when I open settings.",
    "There are three purchases on my card I never made.",
    "Can I get last month's invoice as a PDF?",
    "Je n'arrive pas à me connecter depuis la mise à jour.",
]
team = {
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this message?",
        "criteria": {
            "billing": "Charges, refunds, invoices",
            "technical": "App or site faults",
            "fraud": "Suspected unauthorised use",
            "shipping": "Deliveries, tracking, returns",
            "account": "Login, password, profile settings",
        },
    }
}

# Warm-up the new (state length, 5-way choice) shape.
engine.decide(SystemOneRequest(state=tickets[0], questions=team))
torch.cuda.synchronize()
t0 = time.perf_counter()
results = []
for ticket in tickets:
    resp, meta = engine.decide(SystemOneRequest(state=ticket, questions=team))
    results.append((resp, meta))
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0

for ticket, (resp, _meta) in zip(tickets, results):
    a = resp.answers["team"]
    probs = ", ".join(f"{k} {p:.2f}" for k, p in a.probabilities.items())
    print(f"{a.choice:>9} ({a.confidence:.2f})  {ticket}")
    print(f"{'':>12}[{probs}]")
print(f"\n{len(tickets)} decisions in {elapsed * 1000:.0f} ms ({elapsed / len(tickets) * 1000:.0f} ms / ticket)")
vram_line("batch ")


## 5. Image state (same COCO frame as d1-3B)

d1-3B takes `images=[...]` with `state=None`. elephant-1's **official `qev.DecisionEngine`** (model card
example and the public Space) loads `Qwen3_5ForCausalLM` — text only. The technical report says the
vision encoder was frozen and they trained on text, with image/video training planned.

This cell downloads the same two-cats-on-a-sofa frame the d1 notebook uses, displays it, and tries to
pass pixels through `decide()`. If the runtime has no image path (expected today), it skips rather
than inventing a second protocol, and prints the questions so you can compare when image serving lands.


In [ ]:
from IPython.display import display

vision_questions = {
    "cats": {
        "type": "choice",
        "instructions": "How many cats are there?",
        "criteria": {"one": "One", "two": "Two", "more": "Three or more"},
    },
    "remote": {
        "type": "noul",
        "instructions": "Is there a remote control in the picture?",
    },
    "awake": {
        "type": "noul",
        "instructions": "Are all the cats awake?",
    },
}

image = None
try:
    from transformers.image_utils import load_image

    image = load_image("http://images.cocodataset.org/val2017/000000039769.jpg")
    display(image.resize((320, 240)))
except Exception as exc:
    print("Could not download the COCO sample:", type(exc).__name__, exc)

IMAGE_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"
image_ok = False

if image is not None:
    # Probe the official API only. Do not build an unofficial VL forward.
    decide_params = getattr(engine.decide, "__code__", None)
    accepts_images = bool(decide_params and "images" in decide_params.co_varnames)
    if accepts_images:
        try:
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            vis_resp, vis_meta = engine.decide(
                SystemOneRequest(state=None, questions=vision_questions),
                images=[image],
            )
            torch.cuda.synchronize()
            show(vis_resp, vis_meta, (time.perf_counter() - t0) * 1000)
            image_ok = True
        except TypeError as exc:
            print("decide() rejected images:", exc)
        except Exception as exc:
            print("Image decide() failed:", type(exc).__name__, exc)
    else:
        print(
            "Official qev.DecisionEngine.decide(request) has no images argument "
            "(text-only Qwen3_5ForCausalLM, matching the model-card example and the Space)."
        )

if not image_ok:
    print("Skipping pixel-level image questions so we stay on the official System One path.")
    print("Questions (same as d1-3B) for when image serving lands:")
    print(json.dumps(vision_questions, indent=2))
    print("Sample image:", IMAGE_URL)

vram_line("after image cell ")
print(f"\npeak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")


## Next steps

- **Your own decisions:** swap in your labels and descriptions; descriptive `criteria` help more than long instructions.
- **Shared prefix:** put every question about one state in the same `SystemOneRequest`. Each extra question is only a suffix forward.
- **Larger GPU:** on 24 GB+ with native bfloat16, set the GPU-check cell's 24 GB branch (card default, no 4-bit). `fork_mode="batched"` can score suffixes together after a verified cache fork.
- **Bigger sibling:** [`mammoth-1`](https://huggingface.co/baselabs/mammoth-1) (27B) is the card's highest-accuracy model; it is not a T4 fit.
- **Compare:** rerun the same tickets in [`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb) (3.1B, float16, ~6.2 GB).
- **Licence:** elephant-1 is [MIT](https://huggingface.co/baselabs/elephant-1). The `qev` runtime downloaded above is [Apache 2.0](https://github.com/OmniJev/OneJev).
